In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Load and prepare the data
df = pd.read_csv('../data/processed/pv_weather.csv', parse_dates=['LocalTime'])
df.set_index('LocalTime', inplace=True)
df = df.asfreq('H')  # Ensure hourly frequency

# For univariate forecasting, we'll use only the target column.
data = df[['Power(MW)_ActualEnergy']].copy()

# 2. Scale the data to [0,1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data.values)

# 3. Create sequences from the scaled data
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 24  # Use the past 24 hours to predict the next hour
X_all, y_all = create_sequences(scaled_data, window_size)
# X_all shape: (N, window_size, 1), y_all shape: (N, 1)

# Convert the numpy arrays to torch tensors
X_all = torch.tensor(X_all, dtype=torch.float32)
y_all = torch.tensor(y_all, dtype=torch.float32)

# 4. Split into training and testing sets (80% training)
split_index = int(len(X_all) * 0.8)
X_train = X_all[:split_index]
y_train = y_all[:split_index]
X_test = X_all[split_index:]
y_test = y_all[split_index:]

# 5. Define the LSTM model in PyTorch
class LSTMForecast(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1):
        super(LSTMForecast, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # Initialize hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))  # out: (batch, seq_length, hidden_size)
        # Take the output from the last time step
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = LSTMForecast(input_size=1, hidden_size=50, num_layers=1)

# 6. Train the LSTM model
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50
batch_size = 32

# Create DataLoader for training
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_X.size(0)
    epoch_loss /= len(train_dataset)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}")

# 7. Iterative forecasting for the next 7 days (168 hours)
forecast_horizon = 168
model.eval()

# Use the last window from the scaled data as the starting point
last_window = torch.tensor(scaled_data[-window_size:], dtype=torch.float32).unsqueeze(0)  # shape: (1, window_size, 1)
forecast_scaled = []

with torch.no_grad():
    current_window = last_window.clone()
    for i in range(forecast_horizon):
        # Predict the next time step
        pred = model(current_window)  # shape: (1, 1)
        forecast_scaled.append(pred.item())
        # Update the window: remove the first value and append the prediction
        new_window = current_window.squeeze(0).numpy()  # shape: (window_size, 1)
        new_window = np.vstack((new_window[1:], [[pred.item()]]))
        current_window = torch.tensor(new_window, dtype=torch.float32).unsqueeze(0)

forecast_scaled = np.array(forecast_scaled).reshape(-1, 1)
# Invert the scaling to obtain values in the original scale
forecast = scaler.inverse_transform(forecast_scaled)

# Create a datetime index for the forecast period
last_timestamp = df.index[-1]
future_dates = pd.date_range(start=last_timestamp + pd.Timedelta(hours=1), periods=forecast_horizon, freq='H')

# 8. Plot the forecast along with recent actual data
plt.figure(figsize=(12,6))
# Plot the last 7 days of actual data for context
plt.plot(df['Power(MW)_ActualEnergy'].iloc[-(24*7):], label="Recent Actual")
plt.plot(future_dates, forecast, label="7-Day Forecast", color='red')
plt.title("7-Day Energy Forecast with LSTM (PyTorch)")
plt.xlabel("Time")
plt.ylabel("Power (MW)")
plt.legend()
plt.show()


ModuleNotFoundError: No module named 'torch'

In [ ]:
# 1. Load and prepare the data
df = pd.read_csv('../data/processed/pv_weather.csv', parse_dates=['LocalTime'])
df.set_index('LocalTime', inplace=True)
df = df.asfreq('H')  # Ensure hourly frequency

# For simplicity, use only the target variable for univariate forecasting
data = df[['Power(MW)_ActualEnergy']].copy()



In [ ]:
# 2. Scale the data to the [0,1] range
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data.values)



In [ ]:
# 3. Create sequences for the LSTM
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 24  # Use the past 24 hours to predict the next hour
X_all, y_all = create_sequences(scaled_data, window_size)

# 4. Split the sequences into training and testing sets (80% training, 20% testing)
split_index = int(len(X_all) * 0.8)
X_train, X_test = X_all[:split_index], X_all[split_index:]
y_train, y_test = y_all[:split_index], y_all[split_index:]

# 5. Build and train the LSTM model
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(window_size, 1)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

# Use early stopping to prevent overfitting
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1, callbacks=[es])

# Optional: Evaluate the model on the test set (this RMSE is in scaled units)
pred_test = model.predict(X_test)
test_rmse = np.sqrt(np.mean((scaler.inverse_transform(pred_test) - scaler.inverse_transform(y_test))**2))
print("Test RMSE:", test_rmse)

# 6. Iterative Forecasting for the next 7 days (168 hours)
forecast_horizon = 168
# Use the last available window from the scaled data as the starting input for prediction
last_window = scaled_data[-window_size:].reshape(1, window_size, 1)

forecast_scaled = []
current_window = last_window.copy()

for i in range(forecast_horizon):
    # Predict the next value based on the current window
    pred = model.predict(current_window)
    forecast_scaled.append(pred[0, 0])
    
    # Update the window: remove the oldest value and append the new prediction
    current_window = np.append(current_window[:,1:,:], [[pred]], axis=1)

forecast_scaled = np.array(forecast_scaled).reshape(-1, 1)
# Invert the scaling to obtain forecast values in the original scale
forecast = scaler.inverse_transform(forecast_scaled)

# Create a datetime index for the forecasted period
last_timestamp = df.index[-1]
future_dates = pd.date_range(start=last_timestamp + pd.Timedelta(hours=1), periods=forecast_horizon, freq='H')

# 7. Plot the forecast along with recent actual data for context
plt.figure(figsize=(12,6))
# Plot the last 7 days of actual data
plt.plot(df['Power(MW)_ActualEnergy'].iloc[-(24*7):], label='Recent Actual')
plt.plot(future_dates, forecast, label='7-Day Forecast', color='red')
plt.title('7-Day Energy Forecast with LSTM')
plt.xlabel('Time')
plt.ylabel('Power (MW)')
plt.legend()
plt.show()